<a href="https://colab.research.google.com/github/sebaannkuruvilla/AutomatedEquivalenceTesting/blob/main/Part2_TestDataGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TestData Generator:**

* Input is a Graph
* Data is created  for each Test Scenario and  dumped into a json file


Assumptions:
Vertices are all numbers




In [14]:
import copy
import random
import json
import pprint

import heapq
from collections import deque

In [15]:
def print_json(test_data,filename):
    with open(filename, "w") as f:
        json.dump(test_data, f, indent=4) # Using indent for readability

#Find reachable vertices using DFS
def dfs(graph,source,visited,reachable):
    visited[source]=1
    reachable.append(source)
    for nbor in graph[source]:
        if(visited[nbor]==0):
           visited[nbor]=1
           dfs(graph,nbor,visited,reachable)

def find_reachable_vertices(source,vertex_list,graph):
    visited={i:0 for i in vertex_list} #visited array
    reachable=[] #nodes reachable from a source
    dfs(graph,source,visited,reachable)
    return reachable



#Create Disconnceted Component
def create_component(new_nodes):
    v1,v2,v3=new_nodes
    comp={v1: {v2:6  , v3:5 },
          v2: {},
          v3: {}}

    return (comp,v1)


#Create 2 Graphs for metamorphic testing
#Created from original graph by add the component as disconnected
def add_disc_component(graph_c1,dc_c1):
    graph_c1.update(dc_c1)
    return graph_c1

#reated from original graph by merging the component at point cp
def merge_disc_component(graph_c2,reachable_vertices,dc_c2,connect_point):
    vertex_pick=random.choice(reachable_vertices)
    graph_c2.update(dc_c2)
    graph_c2[vertex_pick][connect_point]=4
    return graph_c2,vertex_pick

def dijkstra_allsp(source,graph):

    distance_dict={}
    min_dist_heap=[]

    for node, nbors in graph.items():
        distance_dict[node]=(float('inf'),[-1]) #(min_dist,parent)

    distance_dict[source]=(0,[-1])  #<current_min_dist, parent>
    heapq.heappush(min_dist_heap,(distance_dict[source][0],source))  #HeapEntry: <distance,vertex>


    while(min_dist_heap):  #while heap not empty
        curr_min,curr_vertex=heapq.heappop(min_dist_heap) # Pop minimum value entry  --node x is reachable at distance ditsx (distx,node)
        if(curr_min>distance_dict[curr_vertex][0]): #check if it qualifies for propagation
            continue

        #If Qualified for propagation
        for nb, wt in graph[curr_vertex].items(): #update neigbours
            if(curr_min+wt<distance_dict[nb][0]): # if we get a strictly lesser value for a neigbour
                  distance_dict[nb]=(curr_min+wt,[curr_vertex])  #update the neighbours distance dict
                  heapq.heappush(min_dist_heap,(curr_min+wt,nb))  #Add neighbour entry to heap
            elif(curr_min+wt==distance_dict[nb][0]): #if an equally good path comes up
                 distance_dict[nb][1].append(curr_vertex)


    return distance_dict


def get_parent_dict_from_source(distances):
    parent_dict={}
    for k,v in distances.items():
        parent_dict[k]=v[1]
    return parent_dict

def find_all_paths_from_source(current_node,path_list,all_sp,parent_dict,source,target): #works for the source for which we ran dijkstra

       if(current_node==source):
          path_list.append(source)
          all_sp.append(copy.deepcopy(path_list))
          return

       path_list.append(current_node)
       parents=parent_dict[current_node]
       for p in parents:
           current_node=p
           find_all_paths_from_source(current_node,path_list,all_sp,parent_dict,source,target)
           path_list.pop()


def find_path(dest,parent):
    prev=dest
    path=[]
    while prev!=-1:
        path.append(str(prev))
        prev=parent[prev]

    return path

def weighted_bfs_orig(graph,source):

    parent={}
    distance={}
    for node, edges in graph.items():
        parent[node]=-1
        distance[node]=float('inf')


    Q=deque()
    distance[source]=0
    parent[source]=-1
    Q.append(source)

    while Q:
      node=Q.popleft()
      for nb, wt in graph[node].items():
          if(distance[nb]==float('inf')):  #new
            distance[nb]=distance[node]+wt
            parent[nb]=node
            Q.append(nb)


    return  distance,parent


* TestCase 0

In [16]:
def testdata_0_compare_op(graph):
    g_all_one=copy.deepcopy(graph)
    for node, nbors in g_all_one.items():
        for nb in nbors:
            g_all_one[node][nb]=1

    vertex_list=[v for v in graph.keys()]

    test_data={ "case_id":0,
                "source_list":vertex_list,
                "graph1":graph,
                "graph2":g_all_one }
    filename="Test0.json"
    print_json(test_data,filename)
    return test_data
    #del test_data


* Test Scenario 1 : Basic Differential Testing

In [17]:
def testdata_1_disconnected_comp(graph):
      vertex_list=[v for v in graph.keys()]
      source_pick=random.choice(vertex_list)  #pick a source
      #source_pick=0 # #****************** remove******************************



      #Create 3 new nodes not in graph
      label_start=max(vertex_list)+1
      label_end=label_start+3
      new_nodes=range(label_start, label_end)

      #Find component and connectiong point using new nodes <Currently fixed structure>
      component,cp=create_component(new_nodes)

      #Find reachable vertices from picked source
      reachable_vertices=find_reachable_vertices(source_pick,vertex_list,graph)

      # Graph_a: created from original graph by add the component as disconnected
      graph_a=add_disc_component(copy.deepcopy(graph),copy.deepcopy(component))


      #Graph_a: created from original graph by merging the component at point cp
      graph_b,vp=merge_disc_component(copy.deepcopy(graph),reachable_vertices,copy.deepcopy(component),cp)


      #Preparing Test data
      test_data={"case_id":2,
                 "graph":graph,
                 "source": source_pick,
                 "rv":reachable_vertices,
                 "comp":component,
                 "connect_point":cp,
                 "vertex_pick":vp,
                 "graph1":graph_a,
                 "graph2":graph_b }

      filename="Test1.json"
      print_json(test_data,filename)



      return test_data

* Test Scenario 2 : Testing subpaths of shortest paths are shortest paths

In [18]:
def testdata_2_spInsp(graph): #2) Data to verify MR in WBFS
    vertex_list=[v for v in graph.keys()]
    source_pick=random.choice(vertex_list)
    #source_pick=0 #****************** remove******************************
    reachable_vertices=find_reachable_vertices(source_pick,vertex_list,graph)
    reachable_vertices.remove(source_pick)
    target_pick=random.choice(reachable_vertices)
    #target_pick=5 #****************** remove******************************

    #1)Data to verify MR in dijkstra
    dij_dist=dijkstra_allsp(source_pick,graph)
    parent_dict=get_parent_dict_from_source(dij_dist)
    all_sp,path_list=[],[]
    find_all_paths_from_source(target_pick,path_list,all_sp,parent_dict,source_pick,target_pick)


    #Data to Verify MR in WBFS
    dist_bfs,parents_bfs=weighted_bfs_orig(graph,source_pick)
    bfs_path=find_path(target_pick,parents_bfs)

    test_data={  "case_id":2,
                 "graph":graph,
                 "source": source_pick,
                 "target": target_pick,
                 "dij_all_sp":all_sp,
                 "bfs_path":bfs_path
                  }
    filename="Test2.json"
    print_json(test_data,filename)

    return test_data

* TestScenario 3: Test how Solvers reach to uniform changes in graph weights

In [19]:
def test_uniform_changes(graph):
  graph_change=copy.deepcopy(graph)
  #add_const=random.randint(2, 20)
  add_const=10
  vertex_list=[vtx for vtx in graph.keys()]
  for node, nbors in graph.items():
      for k, v in nbors.items():
            graph_change[node][k]=v+add_const

  test_data={  "case_id":3,
                 "graph":graph,
                 "graph_change": graph_change,
                 "vertex_list":vertex_list

            }
  filename="Test3.json"
  print_json(test_data,filename)

  return test_data


* TestScenario 4: SP via new nodes

In [20]:
def test_addNodeInSP(graph):
  vertex_list=[vtx for vtx in graph.keys()]
  source=random.choice(vertex_list)
  vertex_list.remove(source)
  target=random.choice(vertex_list)
  vertex_list.remove(target)

  test_data={    "case_id":4,
                  "graph":graph,
                  "source": source,
                  "target": target,
                  "test_nodes":vertex_list

                    }
  filename="Test4.json"
  print_json(test_data,filename)
  return test_data


* TestScenario 5 : Remove nodes in SP

In [21]:
def test_delNode_SP(graph):
    vertex_list=[vtx for vtx in graph.keys()]
    source_pick=random.choice(vertex_list)
    reachable_vertices=find_reachable_vertices(source_pick,vertex_list,graph)
    reachable_vertices.remove(source_pick)
    target_pick=random.choice(reachable_vertices)

    #Dij: get all shortest paths from source to target
    dij_dist=dijkstra_allsp(source_pick,graph)
    parent_dict=get_parent_dict_from_source(dij_dist)
    all_sp,path_list=[],[]
    find_all_paths_from_source(target_pick,path_list,all_sp,parent_dict,source_pick,target_pick)

    #WBFS: get  shortest paths from source to target
    dist_bfs,parents_bfs=weighted_bfs_orig(graph,source_pick)
    bfs_path=find_path(target_pick,parents_bfs)

    test_data={  "case_id":5,
                 "graph":graph,
                 "source": source_pick,
                 "target": target_pick,
                 "dij_all_sp":all_sp,
                 "bfs_path":bfs_path
                  }

    filename="Test5.json"
    print_json(test_data,filename)

    return test_data


* TestScenario 6 : SP after reversing graph

In [22]:
def test_reverse_graph(graph):
    graph_rev={}
    vertex_list=[vtx for vtx in graph.keys()]
    for vtx in vertex_list:
        graph_rev[vtx]={}

    for node, nbors in graph.items():
        for k, v in nbors.items():
              graph_rev[k][node]=v

    test_data={    "case_id":6,
                  "graph":graph,
                  "reverse_graph": graph_rev,
                  "vertex_list":vertex_list

                    }
    filename="Test6.json"
    print_json(test_data,filename)

    return test_data

# Sample Test Data generation

In [29]:
#Part1 Input
p1_graph={ 0: {1:5  , 2:10 },
        1: {3:13 , 4:40 },
        2: {3:15 , 5:20 },
        3: {4:10 , 5:2},
        4: {},
        5: {}}
p2_graph={ 0: {1:10  , 2:10 },
        1: {3:3 , 4:5 },
        2: {3:3 , 5:5 },
        3: {4:2 , 5:2},
        4: {},
        5: {}}
p3_graph={ 0: {1:2  , 2:2 },
        1: {3:4 , 4:3 },
        2: {3:3 , 4:4 },
        3: {5:7},
        4: {6:8},
        5: {},
        6: {}}

test_scenario_list=["Test0:Compare_Outputs","Test1:Disc_Components","Test2:SpInSp","Test3:UniformChange","Test4:AddNodeInSP","Test5:DelNodeInSP","Test6:ReverseGraph"]

In [34]:
def generate_test_data(p1_graph,p2_graph):
    for test in test_scenario_list:
        if(test == "Test0:Compare_Outputs"):
            test_data0=testdata_0_compare_op(p1_graph)
        elif(test =="Test1:Disc_Components"):
            test_data1=testdata_1_disconnected_comp(p1_graph)
        elif(test =="Test2:SpInSp"):
            test_data2=testdata_2_spInsp(p3_graph)
        if(test == "Test3:UniformChange"):
            test_data3=test_uniform_changes(p1_graph)
        elif(test =="Test4:AddNodeInSP"):
            test_data4=test_addNodeInSP(p1_graph)
        elif(test =="Test5:DelNodeInSP"):
            test_data5=test_delNode_SP(p3_graph)
        elif(test =="Test6:ReverseGraph"):
            test_data6=test_reverse_graph(p1_graph)


    return test_data0,test_data1,test_data2,test_data3,test_data4,test_data5,test_data6

In [35]:
print("**********************************************Generating Sample Test data**********************************************")
generate_test_data(p1_graph,p2_graph)

**********************************************Generating Sample Test data**********************************************


({'case_id': 0,
  'source_list': [0, 1, 2, 3, 4, 5],
  'graph1': {0: {1: 5, 2: 10},
   1: {3: 13, 4: 40},
   2: {3: 15, 5: 20},
   3: {4: 10, 5: 2},
   4: {},
   5: {}},
  'graph2': {0: {1: 1, 2: 1},
   1: {3: 1, 4: 1},
   2: {3: 1, 5: 1},
   3: {4: 1, 5: 1},
   4: {},
   5: {}}},
 {'case_id': 2,
  'graph': {0: {1: 5, 2: 10},
   1: {3: 13, 4: 40},
   2: {3: 15, 5: 20},
   3: {4: 10, 5: 2},
   4: {},
   5: {}},
  'source': 3,
  'rv': [3, 4, 5],
  'comp': {6: {7: 6, 8: 5}, 7: {}, 8: {}},
  'connect_point': 6,
  'vertex_pick': 3,
  'graph1': {0: {1: 5, 2: 10},
   1: {3: 13, 4: 40},
   2: {3: 15, 5: 20},
   3: {4: 10, 5: 2},
   4: {},
   5: {},
   6: {7: 6, 8: 5},
   7: {},
   8: {}},
  'graph2': {0: {1: 5, 2: 10},
   1: {3: 13, 4: 40},
   2: {3: 15, 5: 20},
   3: {4: 10, 5: 2, 6: 4},
   4: {},
   5: {},
   6: {7: 6, 8: 5},
   7: {},
   8: {}}},
 {'case_id': 2,
  'graph': {0: {1: 2, 2: 2},
   1: {3: 4, 4: 3},
   2: {3: 3, 4: 4},
   3: {5: 7},
   4: {6: 8},
   5: {},
   6: {}},
  'source': 